In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np

# Ingest OBD and sensor data (assume file names)
obd_train = pd.read_csv('obd_train.csv')
obd_test = pd.read_csv('obd_test.csv')
sensor_train = pd.read_csv('sensor_train.csv')
sensor_test = pd.read_csv('sensor_test.csv')

# Combine OBD and sensor data on a common key if available (e.g., 'timestamp' or 'record_id')
# For demonstration, assume 'record_id' is the common key
train_df = pd.merge(obd_train, sensor_train, on='record_id', how='inner')
test_df = pd.merge(obd_test, sensor_test, on='record_id', how='inner')

# Copy DataFrames for processing
train_proc = train_df.copy()
test_proc = test_df.copy()

# Identify columns
label_col = 'label' if 'label' in train_proc.columns else None
id_col = 'record_id'
feature_cols = [col for col in train_proc.columns if col not in [id_col, label_col]]

# 1. Handle missing values
for col in feature_cols:
    if train_proc[col].dtype in [np.float64, np.int64]:
        # Fill missing with median
        median = train_proc[col].median()
        train_proc[col] = train_proc[col].fillna(median)
        test_proc[col] = test_proc[col].fillna(median)
    else:
        # Fill missing with mode for categorical
        mode = train_proc[col].mode()[0]
        train_proc[col] = train_proc[col].fillna(mode)
        test_proc[col] = test_proc[col].fillna(mode)

# 2. Handle outliers (clip to 1st and 99th percentile for numeric columns)
for col in feature_cols:
    if train_proc[col].dtype in [np.float64, np.int64]:
        lower = train_proc[col].quantile(0.01)
        upper = train_proc[col].quantile(0.99)
        train_proc[col] = train_proc[col].clip(lower, upper)
        test_proc[col] = test_proc[col].clip(lower, upper)

# 3. Normalize numeric features (min-max scaling)
from sklearn.preprocessing import MinMaxScaler

numeric_cols = [col for col in feature_cols if train_proc[col].dtype in [np.float64, np.int64]]
scaler = MinMaxScaler()
train_proc[numeric_cols] = scaler.fit_transform(train_proc[numeric_cols])
test_proc[numeric_cols] = scaler.transform(test_proc[numeric_cols])

# 4. Encode categorical features (ordinal encoding)
from sklearn.preprocessing import OrdinalEncoder

categorical_cols = [col for col in feature_cols if train_proc[col].dtype == object]
if categorical_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_proc[categorical_cols] = encoder.fit_transform(train_proc[categorical_cols])
    test_proc[categorical_cols] = encoder.transform(test_proc[categorical_cols])

# The processed DataFrames are train_proc and test_proc, with labels untouched.
train_proc.head(), test_proc.head()

FileNotFoundError: [Errno 2] No such file or directory: 'obd_train.csv'

In [3]:
import pandas as pd
import numpy as np

# --- Step 1: Generate synthetic OBD and sensor data for train and test ---

# Synthetic OBD data
obd_train = pd.DataFrame({
    'record_id': range(1000, 1050),
    'engine_temp': np.random.normal(90, 10, 50),
    'rpm': np.random.normal(2000, 300, 50),
    'fuel_type': np.random.choice(['petrol', 'diesel', 'electric'], 50),
    'label': np.random.choice([0, 1], 50)  # 0: normal, 1: fault
})

obd_test = pd.DataFrame({
    'record_id': range(1050, 1070),
    'engine_temp': np.random.normal(92, 11, 20),
    'rpm': np.random.normal(2100, 320, 20),
    'fuel_type': np.random.choice(['petrol', 'diesel', 'electric'], 20),
    'label': np.random.choice([0, 1], 20)
})

# Synthetic sensor data
sensor_train = pd.DataFrame({
    'record_id': range(1000, 1050),
    'speed': np.random.normal(60, 15, 50),
    'ambient_temp': np.random.normal(25, 5, 50),
    'sensor_status': np.random.choice(['ok', 'fail', 'warn'], 50)
})

sensor_test = pd.DataFrame({
    'record_id': range(1050, 1070),
    'speed': np.random.normal(62, 16, 20),
    'ambient_temp': np.random.normal(26, 6, 20),
    'sensor_status': np.random.choice(['ok', 'fail', 'warn'], 20)
})

# --- Step 2: Merge OBD and sensor data on 'record_id' ---
train_df = pd.merge(obd_train, sensor_train, on='record_id', how='inner')
test_df = pd.merge(obd_test, sensor_test, on='record_id', how='inner')

# --- Step 3: Copy DataFrames for processing ---
train_proc = train_df.copy()
test_proc = test_df.copy()

# --- Step 4: Identify columns ---
label_col = 'label'
id_col = 'record_id'
feature_cols = [col for col in train_proc.columns if col not in [id_col, label_col]]

# --- Step 5: Handle missing values ---
for col in feature_cols:
    if pd.api.types.is_numeric_dtype(train_proc[col]):
        # Fill missing with median
        median = train_proc[col].median()
        train_proc[col] = train_proc[col].fillna(median)
        test_proc[col] = test_proc[col].fillna(median)
    else:
        # Fill missing with mode for categorical
        mode = train_proc[col].mode()[0]
        train_proc[col] = train_proc[col].fillna(mode)
        test_proc[col] = test_proc[col].fillna(mode)

# --- Step 6: Handle outliers (clip to 1st and 99th percentile for numeric columns) ---
for col in feature_cols:
    if pd.api.types.is_numeric_dtype(train_proc[col]):
        lower = train_proc[col].quantile(0.01)
        upper = train_proc[col].quantile(0.99)
        train_proc[col] = train_proc[col].clip(lower, upper)
        test_proc[col] = test_proc[col].clip(lower, upper)

# --- Step 7: Normalize numeric features (min-max scaling) ---
from sklearn.preprocessing import MinMaxScaler

numeric_cols = [col for col in feature_cols if pd.api.types.is_numeric_dtype(train_proc[col])]
scaler = MinMaxScaler()
train_proc[numeric_cols] = scaler.fit_transform(train_proc[numeric_cols])
test_proc[numeric_cols] = scaler.transform(test_proc[numeric_cols])

# --- Step 8: Encode categorical features (ordinal encoding) ---
from sklearn.preprocessing import OrdinalEncoder

categorical_cols = [col for col in feature_cols if pd.api.types.is_object_dtype(train_proc[col])]
if categorical_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_proc[categorical_cols] = encoder.fit_transform(train_proc[categorical_cols])
    test_proc[categorical_cols] = encoder.transform(test_proc[categorical_cols])

# --- Step 9: Show processed DataFrames (labels untouched) ---
print("Processed train data sample:")
print(train_proc.head())
print("\nProcessed test data sample:")
print(test_proc.head())

Processed train data sample:
   record_id  engine_temp       rpm  fuel_type  label     speed  ambient_temp  \
0       1000     0.560875  0.677829        2.0      1  0.000000      0.680615   
1       1001     0.350146  0.582592        2.0      0  0.490202      0.835432   
2       1002     0.568103  0.576636        2.0      1  0.807114      0.286680   
3       1003     0.294778  0.283668        0.0      1  0.237977      1.000000   
4       1004     0.729026  0.434092        2.0      0  0.205525      0.247510   

   sensor_status  
0            0.0  
1            0.0  
2            1.0  
3            2.0  
4            0.0  

Processed test data sample:
   record_id  engine_temp       rpm  fuel_type  label     speed  ambient_temp  \
0       1050     0.416007  0.506226        2.0      0  0.514084      0.518753   
1       1051     0.375593  0.666124        1.0      0  0.502906      0.415716   
2       1052     0.272554  0.780220        2.0      1  0.840287      0.667743   
3       1053     

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_proc)
print("column_info")
print(column_info)

column_info
{'Category': [], 'Numeric': ['record_id', 'engine_temp', 'rpm', 'fuel_type', 'label', 'speed', 'ambient_temp', 'sensor_status'], 'Datetime': [], 'Others': []}


2026-01-13 13:40:25.813 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT
2026-01-13 13:40:25.822 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT


In [5]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_proc)
print("column_info")
print(column_info)

column_info
{'Category': [], 'Numeric': ['record_id', 'engine_temp', 'rpm', 'fuel_type', 'label', 'speed', 'ambient_temp', 'sensor_status'], 'Datetime': [], 'Others': []}


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

# Prepare features and labels
X_train = train_proc.drop(['record_id', 'label'], axis=1)
y_train = train_proc['label']
X_test = test_proc.drop(['record_id', 'label'], axis=1)
y_test = test_proc['label']

# Train RandomForestClassifier with optimized hyperparameters
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)

print("Random Forest Model Performance:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(report)

# Save the trained model for API integration
joblib.dump(rf, "vehicle_diagnostics_rf_model.joblib")

Random Forest Model Performance:
Accuracy: 0.5500
Precision: 0.5556
Recall: 0.5000
F1-score: 0.5263
Confusion Matrix:
[[6 4]
 [5 5]]

Classification Report:
              precision    recall  f1-score   support

           0       0.55      0.60      0.57        10
           1       0.56      0.50      0.53        10

    accuracy                           0.55        20
   macro avg       0.55      0.55      0.55        20
weighted avg       0.55      0.55      0.55        20



['vehicle_diagnostics_rf_model.joblib']